In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.datasets import make_swiss_roll

## Datos de entrenamiento

In [ ]:
def get_batch(batch_size=1000, noise=0.1):
    x, _ = make_swiss_roll(batch_size, noise=noise)
    x = x[:, [0, 2]]
    x = (x - x.mean()) / x.std()
    return torch.tensor(x).float()

In [ ]:
samples = get_batch()
plt.figure(figsize=(3, 3))
plt.scatter(samples[:, 0], samples[:, 1], s=1)
plt.show()

## Red neuronal

In [ ]:
class EnergyFunction(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )
    
    def forward(self, x):
        return self.net(x).squeeze(-1)

## Clase `EBM`

In [ ]:
class EBM:
    def __init__(self, energy_function):
        self.energy_function = energy_function
    
    def langevin_sampling(self, n_samples, n_steps=200, step_size=0.01):
        x = torch.randn(n_samples, 2)
        
        for _ in range(n_steps):
            x.requires_grad_(True)
            energy = self.energy_function(x)
            grad_energy = torch.autograd.grad(outputs=energy.sum(), inputs=[x])[0]
            
            with torch.no_grad():
                noise = torch.randn_like(x)
                x = x - (step_size / 2) * grad_energy + torch.sqrt(torch.tensor(step_size)) * noise
        
        return x
    
    def train(self, n_iters=1000):

        optimizer = optim.Adam(self.energy_function.parameters(), maximize=True)

        for _ in range(n_iters):
            data_batch = get_batch()

            generated_samples = self.langevin_sampling(len(data_batch))
            energy_generated = self.energy_function(generated_samples)
            energy_data = self.energy_function(data_batch)
            
            likelihood_grad = energy_generated.mean() - energy_data.mean()
            
            optimizer.zero_grad()
            likelihood_grad.backward()
            optimizer.step()

## Entrenamiento

In [ ]:
energy_fn = EnergyFunction(input_dim=2)

ebm = EBM(energy_fn)
ebm.train()

## Generación

In [ ]:
samples = ebm.langevin_sampling(n_samples=5000, n_steps=2000, step_size=0.01)
plt.figure(figsize=(3, 3))
plt.scatter(samples[:, 0], samples[:, 1], s=1)
plt.show()

## Función de energía

In [ ]:
x_range = torch.linspace(-3, 3, 100)
y_range = torch.linspace(-3, 3, 100)
xx, yy = torch.meshgrid(x_range, y_range, indexing='ij')
grid = torch.stack([xx.flatten(), yy.flatten()], dim=1)

with torch.no_grad():
    energies = energy_fn(grid).reshape(100, 100)

real_data = get_batch(batch_size=5000)
plt.figure(figsize=(8, 6))
plt.contourf(xx.numpy(), yy.numpy(), energies.numpy(), levels=20, cmap='viridis')
plt.colorbar(label='Energía')
plt.scatter(real_data[:, 0], real_data[:, 1], s=1, c='white', alpha=0.5)
plt.xlabel('x₁')
plt.ylabel('x₂')
plt.title('Energy landscape')
plt.show()